# NSTC Writer 7B Colab Demo

This notebook runs NSTC Writer 7B for Traditional Chinese NSTC-style research proposal drafting.

It automatically detects the available GPU. On the common Colab T4 GPU, it loads the model in 4-bit mode to avoid running out of VRAM. On larger GPUs, it can use bf16/fp16 automatically.

This notebook uses `ericeric777777/NSTC-Writer-7B` by default. Change `MODEL_ID` only if you fork or rename the model repo.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf huggingface_hub

## 2. Configure Model

Use `LOAD_MODE = "auto"` for most users. It selects 4-bit loading on T4-class GPUs and bf16/fp16 on larger GPUs.

In [ ]:
MODEL_ID = "ericeric777777/NSTC-Writer-7B"

# Options: "auto", "4bit", "16bit", "cpu"
LOAD_MODE = "auto"

SYSTEM_PROMPT = "你是熟悉台灣 NSTC 計畫書格式的繁體中文研究寫作助理。回答應正式、清楚、可供研究人員修改使用。"

print("Model:", MODEL_ID)
print("Load mode:", LOAD_MODE)

## 3. Detect Runtime

In [ ]:
import torch

def get_runtime_info():
    if not torch.cuda.is_available():
        return {
            "has_cuda": False,
            "gpu_name": "CPU only",
            "total_vram_gb": 0,
            "major": None,
            "minor": None,
        }

    props = torch.cuda.get_device_properties(0)
    major, minor = torch.cuda.get_device_capability(0)
    return {
        "has_cuda": True,
        "gpu_name": torch.cuda.get_device_name(0),
        "total_vram_gb": round(props.total_memory / 1024**3, 1),
        "major": major,
        "minor": minor,
    }

runtime_info = get_runtime_info()
runtime_info

## 4. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def choose_load_mode(requested_mode, info):
    if requested_mode != "auto":
        return requested_mode
    if not info["has_cuda"]:
        return "cpu"
    if info["total_vram_gb"] < 24:
        return "4bit"
    return "16bit"

selected_mode = choose_load_mode(LOAD_MODE, runtime_info)
print("Selected mode:", selected_mode)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if selected_mode == "4bit":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=quant_config,
        trust_remote_code=True,
    )
elif selected_mode == "16bit":
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=dtype,
        trust_remote_code=True,
    )
elif selected_mode == "cpu":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
else:
    raise ValueError(f"Unsupported LOAD_MODE: {selected_mode}")

print("Loaded:", MODEL_ID)

## 5. Generate Proposal Text

In [ ]:
def generate_nstc_text(user_prompt, max_new_tokens=768, temperature=0.7, top_p=0.9):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    if selected_mode != "cpu":
        inputs = inputs.to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=temperature > 0,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

prompt = """
請根據以下研究主題，撰寫一段 NSTC 計畫摘要：

研究主題：以生成式 AI 輔助科研計畫書撰寫
研究目標：降低研究人員撰寫初稿的時間成本，並提升計畫書架構完整性
請使用正式繁體中文，包含研究背景、方法、預期成果。
"""

print(generate_nstc_text(prompt))

## 6. Conservative Rewrite Mode

Use lower temperature when rewriting existing proposal content.

In [ ]:
rewrite_prompt = """
請將以下文字改寫成正式 NSTC 計畫書語氣，保留原意，不要新增未提供的事實：

我們想做一個 AI 工具，幫助老師更快寫計畫書，也希望可以讓內容比較完整。
"""

print(generate_nstc_text(rewrite_prompt, max_new_tokens=512, temperature=0.3, top_p=0.85))

## 7. Optional Gradio Demo

Run this cell if you want a simple shareable web UI inside Colab.

In [ ]:
!pip install -q gradio

import gradio as gr

def chat_fn(message, temperature, max_new_tokens):
    return generate_nstc_text(
        message,
        max_new_tokens=int(max_new_tokens),
        temperature=float(temperature),
        top_p=0.9,
    )

demo = gr.Interface(
    fn=chat_fn,
    inputs=[
        gr.Textbox(label="請輸入研究主題或計畫書段落", lines=8),
        gr.Slider(0.1, 1.0, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(128, 1536, value=768, step=128, label="Max new tokens"),
    ],
    outputs=gr.Textbox(label="NSTC Writer 7B 輸出", lines=14),
    title="NSTC Writer 7B",
    description="繁體中文 NSTC 計畫書寫作輔助模型。生成內容請務必由研究人員審閱。",
)

demo.launch(share=True)

## Notes

- T4 users should keep `LOAD_MODE = "auto"` or `LOAD_MODE = "4bit"`.
- Larger GPUs such as A100 can use `LOAD_MODE = "16bit"`.
- Always review generated text before official use.
- Do not rely on the model for current NSTC rules, budgets, or compliance requirements.
- Do not enter confidential or personal data unless the runtime and deployment environment are approved for that use.